# Quality-Loss Overfit Sanity Test (Colab)

This notebook runs the quality-loss overfit sanity check on a GPU: with a fixed ISP, train only the residual CNN on one small batch, compare `lr=1e-3` and `lr=1e-4`, and save the full quality-overfit report back to Google Drive.

The loss is the current quality-aligned surrogate:

`-w_ssim * MS_SSIM(Y_pred, Y_ref) - w_vif * VIF_CFA_to_Y(raw_cfa, Y_pred) - w_unique * UNIQUE(rgb_pred) + w_uv * L1(UV_pred, UV_ref)`

NRQM is not part of the training loss. It is computed only for the eval-time composite score.

## Required Google Drive files

Create this folder in Google Drive:

`MyDrive/ISP_colab/quality_overfit/`

Upload these two files there:

- `quality_overfit_bundle.zip`
- `quality_overfit_batch.h5`

The notebook will write outputs to:

`MyDrive/ISP_colab/quality_overfit_outputs/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Configuration

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ISP_colab')
DRIVE_INPUT = DRIVE_ROOT / 'quality_overfit'
DRIVE_OUTPUT = DRIVE_ROOT / 'quality_overfit_outputs'

CODE_ZIP = DRIVE_INPUT / 'quality_overfit_bundle.zip'
BATCH_H5 = DRIVE_INPUT / 'quality_overfit_batch.h5'

LOCAL_REPO = Path('/content/ISP')
LOCAL_H5 = LOCAL_REPO / 'dataset' / 'quality_overfit_batch.h5'

STEPS = 100
LR_VALUES = [1e-3, 1e-4]
BATCH_SIZE = 1
SEED = 42

W_SSIM = 1.0
W_VIF = 0.3
W_UNIQUE = 0.3
W_UV = 0.1

DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
print(f'Drive input folder:  {DRIVE_INPUT}')
print(f'Drive output folder: {DRIVE_OUTPUT}')

## 2. Check the GPU

In [ ]:
import subprocess
import torch

try:
    print(subprocess.check_output([
        'nvidia-smi',
        '--query-gpu=name,memory.total,driver_version',
        '--format=csv'
    ]).decode())
except Exception as exc:
    print(f'nvidia-smi is not available: {exc}')

print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'device: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('Please switch Colab runtime to GPU before running the full quality-overfit sanity check.')

## 3. Install dependencies

In [ ]:
!pip install -q pyiqa tomli h5py tqdm pillow

## 4. Prepare the local Colab workspace

The code and the small HDF5 batch are copied to `/content`, which is faster than reading them directly from Drive.

In [ ]:
import os
import shutil
import zipfile
import time

if not CODE_ZIP.exists():
    raise FileNotFoundError(f'Missing code bundle: {CODE_ZIP}')
if not BATCH_H5.exists():
    raise FileNotFoundError(f'Missing quality-overfit HDF5 batch: {BATCH_H5}')

if LOCAL_REPO.exists():
    print(f'Removing old local workspace: {LOCAL_REPO}')
    shutil.rmtree(LOCAL_REPO)

LOCAL_REPO.mkdir(parents=True, exist_ok=True)
print(f'Unzipping {CODE_ZIP.name}...', end=' ', flush=True)
t0 = time.time()
with zipfile.ZipFile(CODE_ZIP, 'r') as zf:
    zf.extractall(LOCAL_REPO)
print(f'done ({time.time() - t0:.1f}s)')

LOCAL_H5.parent.mkdir(parents=True, exist_ok=True)
print(f'Copying {BATCH_H5.name}...', end=' ', flush=True)
t0 = time.time()
shutil.copy2(BATCH_H5, LOCAL_H5)
print(f'done ({LOCAL_H5.stat().st_size / 1024**2:.2f} MB, {time.time() - t0:.1f}s)')

required = [
    LOCAL_REPO / 'scripts' / 'run_overfit_test.py',
    LOCAL_REPO / 'scripts' / 'run_quality_overfit_test.py',
    LOCAL_REPO / 'isp' / 'training' / 'quality_loss.py',
    LOCAL_REPO / 'metrics' / 'vif.py',
    LOCAL_REPO / 'data' / 'imx623.toml',
    LOCAL_REPO / 'artifacts' / 'baselines' / 'norm_weights.json',
    LOCAL_H5,
]
for path in required:
    print(f'{path.relative_to(LOCAL_REPO)}: {"OK" if path.exists() else "MISSING"}')
    if not path.exists():
        raise FileNotFoundError(path)

print('\nWorkspace is ready.')

## 5. Inspect the overfit batch

In [ ]:
import h5py

with h5py.File(LOCAL_H5, 'r') as f:
    print(f'HDF5: {LOCAL_H5}')
    for key in f.keys():
        print(f'  {key:<10s} shape={f[key].shape} dtype={f[key].dtype}')
    print('attrs:', dict(f.attrs))

## 6. Run the full quality-overfit sanity check

This runs 100 steps for each learning rate.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, '-B', 'scripts/run_overfit_test.py',
    '--h5-path', 'dataset/quality_overfit_batch.h5',
    '--config', 'data/imx623.toml',
    '--norm-weights', 'artifacts/baselines/norm_weights.json',
    '--output-dir', str(DRIVE_OUTPUT),
    '--device', 'cuda',
    '--batch-size', str(BATCH_SIZE),
    '--steps', str(STEPS),
    '--lr-values', *[str(x) for x in LR_VALUES],
    '--seed', str(SEED),
    '--w-ssim', str(W_SSIM),
    '--w-vif', str(W_VIF),
    '--w-unique', str(W_UNIQUE),
    '--w-uv', str(W_UV),
]

print('Running:')
print(' '.join(cmd))
print()

proc = subprocess.Popen(
    cmd,
    cwd=str(LOCAL_REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
try:
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
finally:
    if proc.poll() is None:
        proc.terminate()

print(f'\n[subprocess exit={ret}]')
if ret != 0:
    raise RuntimeError(f'Quality-overfit sanity test failed with exit code {ret}')

## 7. Verify the saved summary

At least one learning rate should satisfy the core sanity checks. `L1_uv` is treated as a color-regularization diagnostic: it should be monitored and kept reasonable, but it is not a hard failure by itself because the target objective is the quality composite.

In [ ]:
import json

summary_path = DRIVE_OUTPUT / 'quality_overfit_summary.json'
history_path = DRIVE_OUTPUT / 'quality_overfit_history.csv'

print(f'Summary: {summary_path}')
print(f'History:  {history_path}')

with open(summary_path, 'r') as f:
    summaries = json.load(f)

core_flags = [
    'loss_decreased',
    'ms_ssim_increased',
    'vif_increased',
    'unique_increased',
    'composite_not_decreased',
    'cnn_grads_present',
]

passing = []
for item in summaries:
    lr = item['lr']
    start = item['start']
    final = item['final']
    ok = all(bool(item.get(flag)) for flag in core_flags)
    l1_uv_delta = final['l1_uv'] - start['l1_uv']
    l1_uv_rel = l1_uv_delta / max(start['l1_uv'], 1e-12)
    passing.append(ok)
    print(f'\nlr={lr:g}: {"PASS" if ok else "CHECK"}')
    print(f'  loss:      {start["loss"]:.6f} -> {final["loss"]:.6f}')
    print(f'  MS-SSIM:   {start["ms_ssim"]:.4f} -> {final["ms_ssim"]:.4f}')
    print(f'  VIF:       {start["vif"]:.4f} -> {final["vif"]:.4f}')
    print(f'  UNIQUE:    {start["unique"]:.4f} -> {final["unique"]:.4f}')
    print(f'  L1_UV:     {start["l1_uv"]:.6f} -> {final["l1_uv"]:.6f}  ({l1_uv_delta:+.6f}, {l1_uv_rel:+.1%})')
    print(f'  composite: {start["composite"]:.4f} -> {final["composite"]:.4f}')
    for flag in core_flags:
        print(f'    {flag:<24s}: {item.get(flag)}')
    print(f'    {"l1_uv_not_increased":<24s}: {item.get("l1_uv_not_increased")}  (diagnostic, not hard gate)')
    if not item.get('l1_uv_not_increased'):
        print('    warning: UV L1 increased. Prefer the lr with the smaller UV increase, or raise W_UV for the final training run if color artifacts are visible.')

if not any(passing):
    raise AssertionError('No learning rate passed the core sanity checks. Review metrics and visual outputs.')

print('\nCore sanity checks passed for at least one learning rate. Review L1_UV and visual outputs before closing the run.')

## 8. Show before/after visualizations

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

viz_dir = DRIVE_OUTPUT / 'viz'
names = ['y_ref.png', 'y_before.png', 'y_after.png', 'u_ref.png', 'u_before.png', 'u_after.png']

plt.figure(figsize=(15, 6))
for i, name in enumerate(names, 1):
    path = viz_dir / name
    if not path.exists():
        print(f'Missing: {path}')
        continue
    plt.subplot(2, 3, i)
    plt.imshow(Image.open(path), cmap='gray', vmin=0, vmax=255)
    plt.title(name)
    plt.axis('off')
plt.tight_layout()
plt.show()

print(f'All outputs are saved in: {DRIVE_OUTPUT}')